In [2]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# ============================================
# PHASE 2 — REFERENCE STATE
# FAST VERSION
# ============================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_PATH = Path("../data/application_train.csv")

# Load only the columns needed for reference monitoring.
reference_columns = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

X_reference = pd.read_csv(
    DATA_PATH,
    usecols=reference_columns
)

# Fix known sentinel
X_reference["DAYS_EMPLOYED"] = X_reference[
    "DAYS_EMPLOYED"
].replace(365243, np.nan)

print("Reference data loaded:", X_reference.shape)

Reference data loaded: (307511, 11)


In [4]:
#Apply the SAME feature engineering

def engineer_features(df):
    data = df.copy()

    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
    data["EMPLOYED_YEARS"] = -data["DAYS_EMPLOYED"] / 365.25

    def safe_divide(a, b):
        return a / b.replace(0, np.nan)

    data["CREDIT_TO_INCOME"] = safe_divide(
        data["AMT_CREDIT"], data["AMT_INCOME_TOTAL"]
    )

    data["ANNUITY_TO_INCOME"] = safe_divide(
        data["AMT_ANNUITY"], data["AMT_INCOME_TOTAL"]
    )

    data["CREDIT_TO_ANNUITY"] = safe_divide(
        data["AMT_CREDIT"], data["AMT_ANNUITY"]
    )

    data["GOODS_TO_CREDIT"] = safe_divide(
        data["AMT_GOODS_PRICE"], data["AMT_CREDIT"]
    )

    data["INCOME_PER_FAMILY_MEMBER"] = safe_divide(
        data["AMT_INCOME_TOTAL"], data["CNT_FAM_MEMBERS"]
    )

    data["INCOME_PER_CHILD"] = (
        data["AMT_INCOME_TOTAL"] /
        (data["CNT_CHILDREN"] + 1)
    )

    external_sources = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]

    data["EXT_SOURCE_MEAN"] = data[
        external_sources
    ].mean(axis=1)

    data["EXT_SOURCE_MAX"] = data[
        external_sources
    ].max(axis=1)

    return data


X_reference_fe = engineer_features(X_reference)

print("Reference after feature engineering:", X_reference_fe.shape)

Reference after feature engineering: (307511, 21)


In [5]:
#Create reference feature statistics

numeric_reference = X_reference_fe.select_dtypes(
    include=["int64", "float64"]
)

reference_distribution = {}

for column in numeric_reference.columns:
    values = numeric_reference[column].dropna()

    reference_distribution[column] = {
        "count": int(values.count()),
        "mean": float(values.mean()),
        "std": float(values.std()),
        "min": float(values.min()),
        "q01": float(values.quantile(0.01)),
        "q05": float(values.quantile(0.05)),
        "q25": float(values.quantile(0.25)),
        "q50": float(values.quantile(0.50)),
        "q75": float(values.quantile(0.75)),
        "q95": float(values.quantile(0.95)),
        "q99": float(values.quantile(0.99)),
        "max": float(values.max())
    }

print(
    "Reference numerical features:",
    len(reference_distribution)
)

Reference numerical features: 21


In [6]:
#save teh refernece rate 

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

with open(
    MODEL_DIR / "reference_distribution.json",
    "w"
) as f:
    json.dump(
        reference_distribution,
        f,
        indent=4
    )

print("Reference distribution saved.")

Reference distribution saved.
